In [4]:
%pip install pandas
import pandas as pd


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
#reading in OATH sanitation fine dataset 

oath = pd.read_csv("../Data/New_DSNY_Sanitation_Oath_Database_20261019.csv",low_memory=False)

In [6]:
#examining column names 
print(oath.columns)

Index(['Ticket Number', 'Violation Date', 'Violation Time', 'Issuing Agency',
       'Charge #1: Code', 'Charge #1: Code Description',
       'Respondent Last Name', 'Violation Location (Borough)',
       'Violation Location (Block No.)', 'Violation Location (Lot No.)',
       'Violation Location (House #)', 'Violation Location (Street Name)',
       'Violation Location (City)', 'Violation Location (State Name)',
       'Violation Location (Zip Code)', 'Violation Location (Floor)',
       'Hearing Status', 'Hearing Result', 'Total Violation Amount',
       'Violation Details', 'Penalty Imposed', 'Balance Due', 'Paid Amount',
       'Compliance Status', 'Charge #1: Infraction Amount',
       'Respondent Address (Borough)', 'Respondent Address (House #)',
       'Respondent Address (Zip Code)', 'Scheduled Hearing Location',
       'Hearing Date', 'Hearing Time', 'Decision Location (Borough)',
       'Decision Date', 'Date Judgment Docketed',
       'Respondent Address or Facility Number(

In [7]:
#I'm interested in debt so I want to look into balance due. I also want to make sure it is the right data type.

oath['Balance Due'].dtypes

#change balance due to float and clean it up a bit 

oath["Balance Due"] = oath["Balance Due"].str.replace("$","",regex=False).str.replace(",","",regex=False)

oath["Balance Due"] = oath["Balance Due"].astype(float)

In [8]:
#I also plan to incorperate fine dates and want to make sure they are actually stored as dates 

oath["Violation Date"].dtypes

oath["Violation Date"] = pd.to_datetime(oath["Violation Date"])

In [9]:
#Now I want to double check the date range of the data I downloaded 
print("Earliest violation:",oath["Violation Date"].min())
print("Latest violation:",oath["Violation Date"].max())

Earliest violation: 2021-01-02 00:00:00
Latest violation: 2026-10-14 00:00:00


In [10]:
#I want to create a five year period so the data is comparable and keep the tickets from the five years 

start_date = "2021-10-14"
end_date = "2026-10-14"

oath_5years = oath[
    (oath["Violation Date"]>= start_date)&
    (oath["Violation Date"]<= end_date)
].copy()


In [11]:
#Since I'm really interested in debt I want to compare all of the financial penalty rows to see what's going on here 

print(oath_5years[["Penalty Imposed","Balance Due","Compliance Status"]].head(10))

#It looks like 

       Penalty Imposed  Balance Due Compliance Status
162381          $75.00         75.0       Penalty Due
162382         $100.00          0.0     All Terms Met
162383          $75.00         75.0       Penalty Due
162384          $75.00          0.0     All Terms Met
162385         $150.00        150.0       Penalty Due
162386         $100.00          0.0     All Terms Met
162387          $75.00          0.0     All Terms Met
162388         $300.00          0.0     All Terms Met
162389         $300.00        300.0       Penalty Due
162390         $100.00          0.0     All Terms Met


In [12]:
#I'm also curious overall how many tickets have an outstanding balance 

print(oath_5years["Compliance Status"].value_counts(dropna=False))

Compliance Status
All Terms Met    1102449
Penalty Due       503774
Name: count, dtype: int64


In [13]:
#Here I want to isolate only the tickets that still have a positive balance 
tickets_in_debt = oath_5years[oath_5years["Balance Due"]>0]


In [14]:
#Now I want to see how many tickets have an outstanding balance and how much total is owed 

print("Unique tickets oustanding balance:",tickets_in_debt["Ticket Number"].nunique())

print("Total oustanding balances: $", round(tickets_in_debt["Balance Due"].sum(), 2))

Unique tickets oustanding balance: 501907
Total oustanding balances: $ 111289069.0


In [15]:
#I'm interested in the size of these properties, specifically how fines are hitting small homeowners so I want to incorperate PLUTO data
#which has information about the size of each lot 

#Here I am reading in my pluto data

pluto = pd.read_csv("../Data/Primary_Land_Use_Tax_Lot_Output_(PLUTO)_20261026.csv",low_memory=False)

In [16]:
#I want to look at the columns here so I can decide how to merge these two datasets 

print(pluto.columns)

#It looks like there isn't a matching column that makes sense but I should be able to create BBL from borough, block, lot in oath data 

Index(['borough', 'Tax block', 'Tax lot', 'community board',
       'census tract 2010', 'cb2010', 'schooldist', 'council district',
       'postcode', 'firecomp',
       ...
       'transitzone', 'geom', 'basempdate', 'dcasdate', 'edesigdate',
       'landmkdate', 'masdate', 'polidate', 'rpaddate', 'zoningdate'],
      dtype='str', length=108)


In [17]:
#First I want to see how borough, block and lot are stored in OATH since a BBL is numbers based 
#BBL is 10 digit structure 1 digit = borough code, 5 digits = tax block, 4 digits = tax lot 
print(oath['Violation Location (Borough)'].unique())
print(oath['Violation Location (Block No.)'].unique())
print(oath['Violation Location (Lot No.)'].unique())

#It looks like block and lot are correctly formatted but borough needs to be converted to numbers 


<StringArray>
['BRONX', 'BROOKLYN', 'STATEN IS', 'MANHATTAN', 'QUEENS', 'NOT NYC', nan]
Length: 7, dtype: str
[ 4837.  3260.  4016. ...  9665. 15530. 13647.]
[  15.   70.   64. ... 1159. 1472.  889.]


In [18]:
#converting borough to numbers 
oath_5years["borough_code"] = oath_5years[
    "Violation Location (Borough)"
].map({
    "MANHATTAN": 1,
    "BRONX": 2,
    "BROOKLYN": 3,
    "QUEENS": 4,
    "STATEN IS": 5
})
print(oath_5years["borough_code"])

162381     3.0
162382     3.0
162383     3.0
162384     3.0
162385     4.0
          ... 
1768599    2.0
1768600    2.0
1768601    3.0
1768602    2.0
1768603    1.0
Name: borough_code, Length: 1606223, dtype: float64


In [19]:
#Making sure block and lot are ints 

oath_5years["Violation Location (Block No.)"] = (oath_5years["Violation Location (Block No.)"].astype("Int64"))

oath_5years["Violation Location (Lot No.)"] = (oath_5years["Violation Location (Lot No.)"].astype("Int64"))

In [20]:
#Creating a BBL column in oath 
oath_5years["BBL"] = oath_5years.apply(
    lambda row: f"{int(row['borough_code'])}{int(row['Violation Location (Block No.)']):05d}{int(row['Violation Location (Lot No.)']):04d}"
    if pd.notna(row["borough_code"])
    and pd.notna(row["Violation Location (Block No.)"])
    and pd.notna(row["Violation Location (Lot No.)"])
    else None,
    axis=1
)

In [21]:
#Making sure it's stored as a string 

pluto["BBL"] = pluto["BBL"].astype(str)


In [22]:
#Merging the two datasets! I've decided to merge on the left because OATH data is important for me to keep 

merged = oath_5years.merge(pluto, on="BBL",how="left")

In [23]:
#I'm interested to see which postcodes have highest outstanding balance
#First I'm checking on the data type of postcode, it is a float, need to convert to text

merged["postcode"].dtype

for table in [merged, pluto]:
    table["postcode"] = table["postcode"].astype("Int64").astype("string")

merged["Violation Location (Zip Code)"] = (
    merged["Violation Location (Zip Code)"].astype("Int64").astype("string")
)

In [24]:
#Now I'm looking at postcodes ranked by oustanding balance 

zip_summary = merged.groupby("postcode")["Balance Due"].sum()
zip_summary = zip_summary.sort_values(ascending=False)
print(zip_summary.head(10))

#Post code 10467 has highest outstanding balance at $2.3 million

postcode
10467    2299456.0
11226    2212652.0
11207    2112217.0
10458    2099930.0
10472    2000266.0
11233    1962951.0
10459    1841217.0
10469    1826286.0
10457    1818475.0
10468    1741898.0
Name: Balance Due, dtype: float64


In [25]:
#Now I want to filter to only zips with oustanding balance 
tickets_in_debt = merged[merged["Balance Due"]>0]
print(tickets_in_debt)

        Ticket Number Violation Date Violation Time        Issuing Agency  \
0          046986579Z     2021-10-14       08:30:00  SANITATION RECYCLING   
2          047228443N     2021-10-14       09:20:00  SANITATION RECYCLING   
4          047228444P     2021-10-14       23:58:00     SANITATION POLICE   
8          047264672P     2021-10-14       08:58:00     SANITATION OTHERS   
12         047023383L     2021-10-14       21:52:00     SANITATION OTHERS   
...               ...            ...            ...                   ...   
1606218    046898517N     2026-10-14       10:01:00  SANITATION RECYCLING   
1606219    049729276N     2026-10-14       10:01:00  SANITATION RECYCLING   
1606220    049767772J     2026-10-14       19:56:00     SANITATION POLICE   
1606221    049948614R     2026-10-14       16:02:00     SANITATION POLICE   
1606222    049939157X     2026-10-14       08:34:00     SANITATION OTHERS   

        Charge #1: Code             Charge #1: Code Description  \
0       

In [26]:
#And look at the number of tickets in each zip with oustanding balance 

zip_debt_counts = tickets_in_debt.groupby("Violation Location (Zip Code)")["Ticket Number"].count()
zip_debt_counts.sort_values(ascending=False).head(10)
print(zip_debt_counts)

Violation Location (Zip Code)
10001    1337
10002    2604
10003    2901
10004      84
10005      19
         ... 
11691    4756
11692    1781
11693     954
11694     374
12345      11
Name: Ticket Number, Length: 188, dtype: int64


In [27]:
#Now I'm curious about the percentage of tickets in each zip with oustanding balance 

all_tickets_by_zip = merged.groupby(
    "postcode"
)["Ticket Number"].nunique()

debt_tickets_by_zip = tickets_in_debt.groupby(
    "postcode"
)["Ticket Number"].nunique()

percent_with_debt = (
    debt_tickets_by_zip / all_tickets_by_zip * 100
)

percent_with_debt.sort_values(ascending=False).head(10)

postcode
10118    100.000000
10122    100.000000
10278    100.000000
11241     66.666667
10170     66.666667
11692     53.297036
10069     51.492537
10474     51.036764
10103     50.000000
10106     50.000000
Name: Ticket Number, dtype: float64

In [28]:
#As I said before I want to examine small homes in particular which would be filtered by units res. Let me first check it's datatype

merged["unitsres"].map(type).value_counts()

#Looks like it is inconsistent let me convert to numbers in both 

for df in [merged, pluto]:
    df["unitsres"] = pd.to_numeric(
        df["unitsres"].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

## and check unitsres types again and see that they're all numbers

merged["unitsres"].map(type).value_counts()

unitsres
<class 'int'>                         1577550
<class 'pandas.api.typing.NAType'>      28673
Name: count, dtype: int64

In [29]:
#I want to examine properties with 1-3 units. This feels like a fair definition of small homes without isolating all the way to single family.

small_res = merged.loc[merged["unitsres"].between(1, 3)].copy()
print(small_res)

        Ticket Number Violation Date Violation Time        Issuing Agency  \
0          046986579Z     2021-10-14       08:30:00  SANITATION RECYCLING   
1          046986578R     2021-10-14       08:22:00  SANITATION RECYCLING   
2          047228443N     2021-10-14       09:20:00  SANITATION RECYCLING   
3          047228442L     2021-10-14       09:12:00  SANITATION RECYCLING   
7          047327661P     2021-10-14       07:46:00     SANITATION OTHERS   
...               ...            ...            ...                   ...   
1606187    046994080J     2026-10-10       08:40:00     SANITATION OTHERS   
1606193    080053365L     2026-10-10       05:53:00     SANITATION OTHERS   
1606197    080101817R     2026-10-11       08:51:00     SANITATION OTHERS   
1606218    046898517N     2026-10-14       10:01:00  SANITATION RECYCLING   
1606219    049729276N     2026-10-14       10:01:00  SANITATION RECYCLING   

        Charge #1: Code                        Charge #1: Code Description 

In [30]:
#Now I want to look at total oustanding fines by zip within this small residential category
small_res_balance = small_res.groupby(
    "postcode"
)["Balance Due"].sum()

small_res_balance.sort_values(ascending=False).head(10)

postcode
10469    1645586.0
11207    1427109.0
11433    1285209.0
10466    1239041.0
11233    1229651.0
10467    1205661.0
11434    1159013.0
11208    1158574.0
10473    1097624.0
10472    1096970.0
Name: Balance Due, dtype: float64

In [31]:
#Now I want to see the top 5 zips with the most unique tickets with oustanding balances

small_res_debt = small_res[small_res["Balance Due"] > 0]

print(small_res_debt.groupby(
    "postcode"
)["Ticket Number"].nunique().sort_values(ascending=False).head(5))

postcode
10469    7221
11207    5846
10466    5385
10467    5100
11434    4923
Name: Ticket Number, dtype: int64


In [32]:
#And the total number of these small residentials within the pluto dataset 

pluto_small_res = pluto.loc[pluto["unitsres"].between(1, 3)].copy()
print(pluto_small_res.shape)

(671640, 108)


In [33]:
#Now I want to see the number of lots this size in each zip
lots_by_zip = pluto_small_res.groupby("postcode")["BBL"].nunique()

print(lots_by_zip.sort_values(ascending=False).head(10))

postcode
10314    19540
11234    18308
10312    17825
10306    15026
11236    13464
11385    10781
11229    10602
10469    10371
11434     9838
10305     9393
Name: BBL, dtype: int64


In [34]:
#And the most affected zips 

affected_by_zip = small_res_debt.groupby("postcode")["BBL"].nunique()
print(affected_by_zip.sort_values(ascending=False).head(10))


postcode
11208    1809
11207    1805
10469    1724
11434    1694
11236    1614
10466    1525
11203    1519
11413    1455
11420    1452
11234    1422
Name: BBL, dtype: int64


In [43]:
#I feel good about my findings now and want to save them to a CSV!

#Converting dollar amounts to numbers 
def to_money(s):
    return pd.to_numeric(s.astype("string").str.replace(r"[$,]", "",regex=True),errors="coerce")

#Summarizing ticket penalties by zip code 
def penalty_zips(df):
    base = to_money(df["Total Violation Amount"])
    imposed = to_money(df["Penalty Imposed"])
    balance = to_money(df["Balance Due"])

    #Tickets where penalty exceeded original fine
    escalated = imposed > base 
    #Escalated tickets that still have oustanding balance
    owing = escalated & (balance>0)

    #Grouping zips by ZIP
    zips = df["postcode"]

    #Calculating totals for each zip code 
    zip_summary = pd.DataFrame({
        "tickets":zips.value_counts(),
        "escalated":escalated.groupby(zips).sum(),
        "escalated_and_owing":owing.groupby(zips).sum(),
        "dollars_added": (imposed - base).clip(lower=0).groupby(zips).sum(),
    })

    #Calculating percentage of tickets that were escalated
    zip_summary["pct_escalated"] = (zip_summary["escalated"]/ zip_summary["tickets"]*100).round(1)

    return zip_summary

#Running for my 1-3 units categorization
zip_summary_1_3 = penalty_zips(small_res)

#Saving the result into a CSV
zip_summary_1_3.to_csv("../Output/penalty_zips_1_3_units.csv")

In [44]:
import os
print(os.getcwd())

/Users/macynhanzlik-barend/Documents/DSNY_Project/Notebooks


In [45]:
import os
print(os.listdir("../Output"))

['penalty_zips_1_3_units.csv']


In [46]:
saved = pd.read_csv("../Output/penalty_zips_1_3_units.csv")

print("Tickets:", saved["tickets"].sum())
print("Escalated:", saved["escalated"].sum())
print("Escalated and owing:", saved["escalated_and_owing"].sum())
print("Dollars added:", saved["dollars_added"].sum())

Tickets: 829247
Escalated: 292216
Escalated and owing: 192317
Dollars added: 62156704.0
